# Amazon Reviews 2023 — 4 categories, 2,000,000 total sample (Colab + Drive)


Goal:
- use these 4 review categories:
  - Electronics
  - Home_and_Kitchen
  - Beauty_and_Personal_Care
  - Sports_and_Outdoors
- build a **balanced sample of 2,000,000 reviews total**

## Why this setup

The official Amazon Reviews'23 documentation lists these review counts:
- Electronics: 43.9M
- Home_and_Kitchen: 67.4M
- Beauty_and_Personal_Care: 23.9M
- Sports_and_Outdoors: 19.6M

So taking **500,000 from each category** gives a balanced **2,000,000-row** dataset without going near full scale.

In [10]:
!pip -q install -U pandas pyarrow requests tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [11]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/amazon_reviews_2023_4cat_rawstream")
RAW_SAMPLE_DIR = PROJECT_DIR / "raw_sample_chunks"
FINAL_DIR = PROJECT_DIR / "final"
LOCAL_DIR = PROJECT_DIR / "local_ready"

for p in [PROJECT_DIR, RAW_SAMPLE_DIR, FINAL_DIR, LOCAL_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Project folder:", PROJECT_DIR)

Project folder: /content/drive/MyDrive/amazon_reviews_2023_4cat_rawstream


In [13]:
RAW_URLS = {
    "Electronics": "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Electronics.jsonl.gz",
    "Home_and_Kitchen": "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Home_and_Kitchen.jsonl.gz",
    "Beauty_and_Personal_Care": "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Beauty_and_Personal_Care.jsonl.gz",
    "Sports_and_Outdoors": "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Sports_and_Outdoors.jsonl.gz",
}

# Official review counts from the dataset page (used only to set a sampling probability).
REVIEW_COUNTS = {
    "Electronics": 43_900_000,
    "Home_and_Kitchen": 67_400_000,
    "Beauty_and_Personal_Care": 23_900_000,
    "Sports_and_Outdoors": 19_600_000,
}

In [14]:
SAMPLE_PER_CATEGORY = 500_000          # 4 * 500k = 2,000,000
LOCAL_SAMPLE_N = 200_000               # small local-ready file
OVERSAMPLE_FACTOR = 1.25               # collect a bit extra, then trim exactly to 500k
WRITE_CHUNK_SIZE = 50_000
SEED = 42

KEEP_COLS = [
    "rating",
    "title",
    "text",
    "parent_asin",
    "user_id",
    "timestamp",
    "helpful_vote",
    "verified_purchase",
]

In [15]:
import gzip
import json
import hashlib
import requests
import pandas as pd
from tqdm.auto import tqdm

In [16]:
def transform_record(record, category):
    row = {k: record.get(k) for k in KEEP_COLS}
    row["category"] = category

    text = row.get("text")
    if text is None:
        text = ""
    text = str(text)
    row["text"] = text
    row["text_len"] = len(text)

    title = row.get("title")
    row["title"] = "" if title is None else str(title)

    row["rating"] = pd.to_numeric(row.get("rating"), errors="coerce")
    row["helpful_vote"] = pd.to_numeric(row.get("helpful_vote"), errors="coerce")
    row["timestamp"] = pd.to_numeric(row.get("timestamp"), errors="coerce")
    row["review_time"] = pd.to_datetime(row["timestamp"], unit="ms", errors="coerce")

    vp = row.get("verified_purchase")
    row["verified_purchase"] = bool(vp) if pd.notna(vp) else False

    return row


def stable_hash_keep_probability(record, keep_prob):
    # Use a deterministic key so reruns are stable.
    key = "||".join([
        str(record.get("user_id", "")),
        str(record.get("parent_asin", "")),
        str(record.get("timestamp", "")),
        str(record.get("rating", "")),
        str(record.get("text", ""))[:200],
    ])
    h = hashlib.blake2b(key.encode("utf-8", errors="ignore"), digest_size=8).digest()
    h_int = int.from_bytes(h, "big")
    score = h_int / float(2**64 - 1)
    return score < keep_prob

In [17]:
def stream_hash_sample_category(category, url, total_reviews, target_n=SAMPLE_PER_CATEGORY,
                                oversample_factor=OVERSAMPLE_FACTOR, chunk_size=WRITE_CHUNK_SIZE):
    keep_prob = min(1.0, oversample_factor * target_n / total_reviews)
    print(f"\n=== Category: {category} ===")
    print("URL:", url)
    print("Approx keep probability:", round(keep_prob, 6))

    out_dir = RAW_SAMPLE_DIR / category
    out_dir.mkdir(parents=True, exist_ok=True)

    buffer = []
    kept = 0
    seen = 0
    chunk_id = 0

    with requests.get(url, stream=True, timeout=120) as resp:
        resp.raise_for_status()
        with gzip.GzipFile(fileobj=resp.raw) as gz:
            for raw_line in gz:
                seen += 1
                rec = json.loads(raw_line)
                row = transform_record(rec, category)

                # basic quality filter
                if row["text_len"] == 0:
                    continue

                if stable_hash_keep_probability(row, keep_prob):
                    buffer.append(row)
                    kept += 1

                if len(buffer) >= chunk_size:
                    out_path = out_dir / f"{category}_accepted_{chunk_id:03d}.parquet"
                    pd.DataFrame(buffer).to_parquet(out_path, index=False)
                    print(f"saved {out_path.name} | seen={seen:,} | kept={kept:,}")
                    buffer = []
                    chunk_id += 1

                if seen % 1_000_000 == 0:
                    print(f"progress | seen={seen:,} | kept={kept:,}")

    if buffer:
        out_path = out_dir / f"{category}_accepted_{chunk_id:03d}.parquet"
        pd.DataFrame(buffer).to_parquet(out_path, index=False)
        print(f"saved {out_path.name} | seen={seen:,} | kept={kept:,}")

    print(f"Finished {category}: seen={seen:,}, kept={kept:,}")
    return {"category": category, "seen": seen, "kept": kept, "keep_prob": keep_prob}

In [18]:
sampling_logs = []
for category, url in RAW_URLS.items():
    log = stream_hash_sample_category(
        category=category,
        url=url,
        total_reviews=REVIEW_COUNTS[category],
        target_n=SAMPLE_PER_CATEGORY
    )
    sampling_logs.append(log)

pd.DataFrame(sampling_logs)


=== Category: Electronics ===
URL: https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Electronics.jsonl.gz
Approx keep probability: 0.014237
progress | seen=1,000,000 | kept=14,265
progress | seen=2,000,000 | kept=28,233
progress | seen=3,000,000 | kept=42,353
saved Electronics_accepted_000.parquet | seen=3,540,521 | kept=50,000
progress | seen=4,000,000 | kept=56,622
progress | seen=5,000,000 | kept=70,731
progress | seen=6,000,000 | kept=84,940
progress | seen=7,000,000 | kept=99,215
saved Electronics_accepted_001.parquet | seen=7,051,822 | kept=100,000
progress | seen=8,000,000 | kept=113,589
progress | seen=9,000,000 | kept=127,761
progress | seen=10,000,000 | kept=141,945
saved Electronics_accepted_002.parquet | seen=10,559,826 | kept=150,000
progress | seen=11,000,000 | kept=156,294
progress | seen=12,000,000 | kept=170,691
progress | seen=13,000,000 | kept=184,961
progress | seen=14,000,000 | kept=199,262
saved Electronics_accepted_003.parquet | 

,category,seen,kept,keep_prob
0,Electronics,43886944,624960,0.014237
1,Home_and_Kitchen,67409944,625150,0.009273
2,Beauty_and_Personal_Care,23911390,625011,0.026151
3,Sports_and_Outdoors,19595170,625559,0.031888


In [19]:
def finalize_exact_category_sample(category, target_n=SAMPLE_PER_CATEGORY, seed=SEED):
    chunk_files = sorted((RAW_SAMPLE_DIR / category).glob("*.parquet"))
    if not chunk_files:
        raise ValueError(f"No chunks found for {category}")

    parts = [pd.read_parquet(f) for f in chunk_files]
    df = pd.concat(parts, ignore_index=True)

    dedup_cols = ["user_id", "parent_asin", "timestamp", "text"]
    df = df.drop_duplicates(subset=dedup_cols).reset_index(drop=True)

    print(category, "accepted rows after dedup:", len(df))

    if len(df) < target_n:
        raise ValueError(
            f"{category} has only {len(df):,} accepted rows, below target {target_n:,}. "
            "Increase OVERSAMPLE_FACTOR and rerun this category."
        )

    out = df.sample(n=target_n, random_state=seed).reset_index(drop=True)

    out_path = FINAL_DIR / f"{category}_sample_{target_n}.parquet"
    out.to_parquet(out_path, index=False)

    print("Saved:", out_path)
    return out_path

In [20]:
category_sample_paths = []
for category in RAW_URLS:
    p = finalize_exact_category_sample(category)
    category_sample_paths.append(p)

category_sample_paths

Electronics accepted rows after dedup: 618111
Saved: /content/drive/MyDrive/amazon_reviews_2023_4cat_rawstream/final/Electronics_sample_500000.parquet
Home_and_Kitchen accepted rows after dedup: 618990
Saved: /content/drive/MyDrive/amazon_reviews_2023_4cat_rawstream/final/Home_and_Kitchen_sample_500000.parquet
Beauty_and_Personal_Care accepted rows after dedup: 618492
Saved: /content/drive/MyDrive/amazon_reviews_2023_4cat_rawstream/final/Beauty_and_Personal_Care_sample_500000.parquet
Sports_and_Outdoors accepted rows after dedup: 618671
Saved: /content/drive/MyDrive/amazon_reviews_2023_4cat_rawstream/final/Sports_and_Outdoors_sample_500000.parquet


[PosixPath('/content/drive/MyDrive/amazon_reviews_2023_4cat_rawstream/final/Electronics_sample_500000.parquet'),
 PosixPath('/content/drive/MyDrive/amazon_reviews_2023_4cat_rawstream/final/Home_and_Kitchen_sample_500000.parquet'),
 PosixPath('/content/drive/MyDrive/amazon_reviews_2023_4cat_rawstream/final/Beauty_and_Personal_Care_sample_500000.parquet'),
 PosixPath('/content/drive/MyDrive/amazon_reviews_2023_4cat_rawstream/final/Sports_and_Outdoors_sample_500000.parquet')]

In [21]:
final_parts = [pd.read_parquet(p) for p in category_sample_paths]
full_df = pd.concat(final_parts, ignore_index=True)

master_path = FINAL_DIR / "amazon_reviews_4cat_2m.parquet"
full_df.to_parquet(master_path, index=False)

print("Saved master file:", master_path)
print("Rows:", len(full_df))
print("Approx file size (GB):", round(master_path.stat().st_size / (1024**3), 3))

Saved master file: /content/drive/MyDrive/amazon_reviews_2023_4cat_rawstream/final/amazon_reviews_4cat_2m.parquet
Rows: 2000000
Approx file size (GB): 0.366


In [22]:
local_df = full_df.sample(n=min(LOCAL_SAMPLE_N, len(full_df)), random_state=SEED).reset_index(drop=True)

local_path = LOCAL_DIR / f"amazon_reviews_4cat_local_{len(local_df)}.parquet"
local_df.to_parquet(local_path, index=False)

print("Saved local-ready file:", local_path)
print("Rows:", len(local_df))
print("Approx file size (GB):", round(local_path.stat().st_size / (1024**3), 3))

Saved local-ready file: /content/drive/MyDrive/amazon_reviews_2023_4cat_rawstream/local_ready/amazon_reviews_4cat_local_200000.parquet
Rows: 200000
Approx file size (GB): 0.038


In [23]:
print(full_df["category"].value_counts())
print()
print(full_df["rating"].value_counts(dropna=False).sort_index())
print()
print(full_df["verified_purchase"].value_counts(dropna=False))

category
Electronics                 500000
Home_and_Kitchen            500000
Beauty_and_Personal_Care    500000
Sports_and_Outdoors         500000
Name: count, dtype: int64

rating
1.0     220319
2.0     101422
3.0     136757
4.0     237296
5.0    1304206
Name: count, dtype: int64

verified_purchase
True     1849843
False     150157
Name: count, dtype: int64


In [24]:
full_df[["rating", "text_len", "helpful_vote"]].describe(include="all")

,rating,text_len,helpful_vote
count,2.000000e+06,2.000000e+06,2.000000e+06
mean,4.151824e+00,1.986957e+02,1.093033e+00
std,1.375395e+00,3.159804e+02,2.120926e+01
min,1.000000e+00,1.000000e+00,-2.000000e+00
25%,4.000000e+00,4.600000e+01,0.000000e+00
50%,5.000000e+00,1.110000e+02,0.000000e+00
75%,5.000000e+00,2.330000e+02,0.000000e+00
max,5.000000e+00,3.186100e+04,1.814700e+04
